# Rnadom Forest Classifier

In [33]:
import time
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold, KFold
import itertools
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    make_scorer
)

## Data Preparation

In [34]:
congressional_df_train_path = "congressional_voting/congressional_df_train_preprocessed.csv"
congresional_df_test_path = "congressional_voting/congressional_df_test_preprocessed.csv"
# obesity_df_train_path = "obesity_df_train_preprocessed.csv"
# obesity_df_test_path = "obesity_df_test_preprocessed.csv"
# depression_df_test_path = "depression_df_test_preprocessed.csv"
# depression_df_train_path = "depression_df_train_preprocessed.csv"
# rev_df_train_path = "rev_df_train_preprocessed.csv"
# rev_df_test_path = "rev_df_test_preprocessed.csv"

# --- Import Data ---
congressional_df_train = pd.read_csv(congressional_df_train_path)
congressional_df_test = pd.read_csv(congresional_df_test_path)
# obesity_df_train = pd.read_csv(obesity_df_train_path)
# obesity_df_test = pd.read_csv(obesity_df_test_path)
# depression_df_train = pd.read_csv(depression_df_train_path)
# depression_df_test = pd.read_csv(depression_df_test_path)
# rev_df_train = pd.read_csv(rev_df_train_path)
# rev_df_test = pd.read_csv(rev_df_test_path)


# train-val split for Houldout Validation
def train_val_split(train_df):
  nrows = train_df.shape[0]

  train_size = int(0.9 * nrows)

  holdout_train_df = train_df[:train_size]
  holdout_val_df = train_df[train_size:]

  return holdout_train_df, holdout_val_df

# obesity_df_train_holdout, obesity_df_val_holdout = train_val_split(obesity_df_train)
# depression_df_train_holdout, depression_df_val_holdout = train_val_split(depression_df_train)
congressional_df_train_holdout, congressional_df_val_holdout = train_val_split(congressional_df_train)
# rev_df_train_holdout, rev_df_val_holdout = train_val_split(rev_df_train)


## Function Definitions

In [38]:
# --- for rfc training with Grid Search and Cross Validation ---
def train_random_forest_with_grid_cv(df, target_attribute, main_scorer):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro'),
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      # 'max_depth': [],  # defined below; acc to all datasets
      'min_samples_split': range(2,10,2),
      'min_samples_leaf': range(1,5,1),
      'max_features': ['sqrt', 'log2'] # after bootstrap, # of features at every split 
  }

  if df.shape[0] <= 1000:
    param_grid["max_depth"] = range(3,8,1)
  elif df.shape[0] > 1000 and df.shape[0] <= 10000:
    param_grid["max_depth"] = range(5,15,1)
  else:
    param_grid["max_depth"] = range(10,25,1)

  # # decision process for max_depth
  # eg:
  #   in congressional data:
  #     218 samples
  #   max_depth => sample_sixe/(2^max_depth); contains [5-10]samples per leaf
  #   so here: 218/(2^5) = 6.8 samples per leaf
  #  => max-depth = log_base2(samples) - c[1,4]

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  rfc = RandomForestClassifier(n_estimators=500, # jus any no bw 500-1000 coz diff datasets may overfit/underfit
                               bootstrap=True,
                               random_state=1, # for reprocucibility sake
                               oob_score=True,
                               n_jobs=-1 # how may processors to use in parallel
                               )

  cv_strategy = KFold(
    n_splits=5,
    shuffle=True,
    random_state=1
  )

  grid_search = GridSearchCV(
      estimator=rfc,
      param_grid=param_grid,
      cv=cv_strategy,
      scoring=scoring,
      refit=main_scorer,
      verbose=True
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  results["Completion_time"] = elapsed
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results, le, grid_search.best_estimator_




# ---  for rfc training with Grid Search and Holdout Validation ---
def train_random_forest_with_grid_holdout(df_train, df_val, target_attribute, main_scorer):

  scoring = {
    'accuracy': accuracy_score,
    'precision': lambda y_true, y_pred: precision_score(y_true, y_pred, average='macro'),
    'recall': lambda y_true, y_pred: recall_score(y_true, y_pred, average='macro'),
    'f1': lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro')
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      # 'max_depth': [],  # defined below; acc to all datasets
      'min_samples_split': range(2,10,2),
      'min_samples_leaf': range(1,5,1),
      'max_features': ['sqrt', 'log2'] # after bootstrap, # of features at every split 
  }

  if df_train.shape[0] <= 1000:
    param_grid["max_depth"] = range(3,8,1)
  elif df_train.shape[0] <= 10000:
    param_grid["max_depth"] = range(5,15,1)
  else:
    param_grid["max_depth"] = range(10,25,1)

  x_train = df_train.loc[:, df_train.columns != target_attribute]
  y_train_raw = df_train[target_attribute]
  le = LabelEncoder()
  y_train = le.fit_transform(y_train_raw)

  x_val = df_val.loc[:, df_val.columns != target_attribute]
  y_val_raw = df_val[target_attribute]
  le = LabelEncoder()
  y_val = le.fit_transform(y_val_raw)

  x_full = pd.concat([x_train, x_val], axis=0)
  y_full = np.concatenate([y_train, y_val])

  start = time.perf_counter()

  param_combinations = list(itertools.product(
    param_grid['criterion'],
    param_grid['min_samples_split'],
    param_grid['min_samples_leaf'],
    param_grid['max_features'],
    param_grid['max_depth']
))

  best_score = 0
  best_model = None
  results_list = []

  for criterion, min_samples_split, min_samples_leaf, max_features, max_depth in param_combinations:
      model = RandomForestClassifier(
          n_estimators=500,
          bootstrap=True,
          oob_score=True,
          random_state=1,
          n_jobs=-1,
          criterion=criterion,
          min_samples_split=min_samples_split,
          min_samples_leaf=min_samples_leaf,
          max_features=max_features,
          max_depth=max_depth
      )
      model.fit(x_train, y_train)
      y_pred = model.predict(x_val)

      result = {
          'param_criterion': criterion,
          'param_min_samples_split': min_samples_split,
          'param_max_depth': max_depth,
          'accuracy': accuracy_score(y_val, y_pred),
          'precision': precision_score(y_val, y_pred, average='macro'),
          'recall': recall_score(y_val, y_pred, average='macro'),
          'f1': f1_score(y_val, y_pred, average='macro')
      }

      results_list.append(result)

      if result[main_scorer] > best_score:
          best_score = result[main_scorer]
          best_model = model

  final_model = RandomForestClassifier(
      criterion=best_model.get_params()['criterion'],
      min_samples_split=best_model.get_params()['min_samples_split'],
      min_samples_leaf=best_model.get_params()['min_samples_leaf'],
      max_features=best_model.get_params()['max_features'],
      max_depth=best_model.get_params()['max_depth'],
      random_state=1
  )

  final_model.fit(x_full, y_full)


  elapsed = time.perf_counter() - start
  results = pd.DataFrame(results_list)
  results['Completion_time'] = elapsed

  print("Best", main_scorer, best_score)
  print("Time(s):", elapsed)
  return results, le, final_model


# --- for getting top 5 results ---
def get_top_results_cv(results_df, main_scorer):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  if f"mean_test_{main_scorer}" in mean_score_metrics:
    mean_score_metrics.remove(f"mean_test_{main_scorer}")
  mean_score_metrics.insert(0, f"mean_test_{main_scorer}")
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant

def get_top_results_holdout(results_df, main_scorer):
  mean_score_metrics = ["f1", "accuracy", "precision", "recall"]
  if main_scorer in mean_score_metrics:
    mean_score_metrics.remove(main_scorer)
  mean_score_metrics.insert(0, main_scorer)
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant


# --- for Prediction on Test Data ---
def pred_test_data(test_df, model, label_encoder, target_attribute, has_ground_truth):
  x_test = test_df.loc[:, test_df.columns != target_attribute]
  x_test_ids = []
  if "ID" in x_test.columns:
    x_test_ids = x_test["ID"]
    x_test = x_test.drop(columns=["ID"])

  if has_ground_truth:
    y_test_raw = test_df[target_attribute]
    y_test = label_encoder.transform(y_test_raw)


  start = time.perf_counter()

  y_pred = model.predict(x_test)

  elapsed = time.perf_counter() - start
  final_results = pd.DataFrame()
  final_results["time"] = [elapsed]
  final_results["parameters"] = [model.get_params()]
  if has_ground_truth:
    final_results["accuracy"] = [accuracy_score(y_test, y_pred)]
    final_results["precision"] = [precision_score(y_test, y_pred, average="macro")]
    final_results["recall"] = [recall_score(y_test, y_pred, average="macro")]
    final_results["f1"] = [f1_score(y_test, y_pred, average="macro")]
  final_pred = x_test.copy()
  final_pred["y_pred"] = label_encoder.inverse_transform(y_pred)
  if len(x_test_ids) > 0:
    final_pred["id"] = x_test_ids

  return final_results, final_pred


## Grid Search - Cross Validation

### Training

In [ ]:
results_obesity_cv, le_obesity_cv, obesity_best_model_cv = train_random_forest_with_grid_cv(obesity_df_train, "obesity_level_grouped", "accuracy")

In [ ]:
results_depression_cv, le_depression_cv, depression_best_model_cv = train_random_forest_with_grid_cv(depression_df_train, "depression", "accuracy")

In [36]:
results_congressional_cv, le_congressional_cv, congressional_best_model_cv = train_random_forest_with_grid_cv(congressional_df_train, "class", "accuracy")

Fitting 5 folds for each of 320 candidates, totalling 1600 fits


Exception in thread ExecutorManagerThread:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py", line 560, in run
    self.terminate_broken(bpe)
  File "/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py", line 752, in terminate_broken
    self.join_executor_internals()
  File "/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py", line 824, in join_executor_internals
    self.shutdown_workers()
  File "/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py", line 790, in shutdown_workers
    with self.processes_management_lock:
  File "/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/backend/synchronize.py", line 115, in __exit__
    return self._semlock.release()
           ^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
results_rev_cv, le_rev_cv, rev_best_model_cv = train_random_forest_with_grid_cv(rev_df_train, "class", "accuracy")

### Top Results

In [ ]:
processed_results_obesity_cv = get_top_results_cv(results_obesity_cv, "accuracy")
display('Top Results for Obesity',processed_results_congressional_cv.head())

In [ ]:
processed_results_depression_cv = get_top_results_cv(results_depression_cv, "accuracy")
display('Top Results for Depression',processed_results_congressional_cv.head())

In [31]:
processed_results_congressional_cv = get_top_results_cv(results_congressional_cv, "accuracy")
display('Top Results for Congressional Voting',processed_results_congressional_cv.head())

'Top Results for Congressional Voting'

,mean_test_accuracy,mean_test_f1,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split
224,0.958668,0.956502,0.957415,0.958073,10,gini,5,sqrt,1,2
240,0.958668,0.956502,0.957415,0.958073,10,gini,5,log2,1,2
66,0.958668,0.956489,0.956384,0.957777,16,entropy,5,sqrt,1,6
67,0.958668,0.956489,0.956384,0.957777,16,entropy,5,sqrt,1,8
82,0.958668,0.956489,0.956384,0.957777,16,entropy,5,log2,1,6


In [ ]:
processed_results_rev_cv = get_top_results_cv(results_rev_cv, "accuracy")
display('Top Results for Amazon Reviews',processed_results_congressional_cv.head())

### Predictions on Test Data

In [ ]:
prediction_results_obesity_cv, y_pred_obesity_cv = pred_test_data(obesity_df_test, obesity_best_model_cv, le_obesity_cv, "obesity_level_grouped", True)
display('Obesity',prediction_results_obesity_cv)

In [ ]:
prediction_results_depression_cv, y_pred_depression_cv = pred_test_data(depression_df_test, depression_best_model_cv, le_depression_cv, "depression", True)
display('Depression',prediction_results_depression_cv)

In [29]:
prediction_results_congressional_cv, y_pred_congressional_cv = pred_test_data(congressional_df_test, congressional_best_model_cv, le_congressional_cv, "class", False)
display('Congressional Voting',prediction_results_congressional_cv)

'Congressional Voting'

,time,parameters
0,0.666503,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."


In [ ]:
prediction_results_rev_cv, y_pred_rev_cv = pred_test_data(rev_df_test, rev_best_model_cv, le_rev_cv, "class", False)
display('Amazon Reviews',prediction_results_rev_cv)

## Grid Search - Holdout

### Training

In [ ]:
results_obesity_holdout, le_obesity_holdout, obesity_best_model_holdout = train_random_forest_with_grid_holdout(obesity_df_train_holdout, obesity_df_val_holdout, "obesity_level_grouped", "accuracy")

In [ ]:
results_depression_holdout, le_depression_holdout, depression_best_model_holdout = train_random_forest_with_grid_holdout(depression_df_train_holdout, depression_df_val_holdout, "depression", "accuracy")

In [39]:
results_congressional_holdout, le_congressional_holdout, congressional_best_model_holdout = train_random_forest_with_grid_holdout(congressional_df_train_holdout, congressional_df_val_holdout, "class", "accuracy")


Best accuracy 1.0
Time(s): 329.94064183399314


In [ ]:
results_rev_holdout, le_rev_holdout, rev_best_model_holdout = train_random_forest_with_grid_holdout(rev_df_train_holdout, rev_df_val_holdout, "class", "accuracy")

### Top Results

In [ ]:
processed_results_obesity_holdout = get_top_results_holdout(results_obesity_holdout, "accuracy")
display('Obesity',processed_results_obesity_holdout.head())

In [ ]:
processed_results_depression_holdout = get_top_results_holdout(results_depression_holdout, "accuracy")
display('Depression',processed_results_depression_holdout.head())

In [40]:
processed_results_congressional_holdout = get_top_results_holdout(results_congressional_holdout, "accuracy")
display('Congressional Voting Holdout results',processed_results_congressional_holdout.head())

'Congressional Voting Holdout results'

,accuracy,f1,precision,recall,param_criterion,param_min_samples_split,param_max_depth
0,1.0,1.0,1.0,1.0,entropy,2,3
5,1.0,1.0,1.0,1.0,entropy,2,3
10,1.0,1.0,1.0,1.0,entropy,2,3
11,1.0,1.0,1.0,1.0,entropy,2,4
15,1.0,1.0,1.0,1.0,entropy,2,3


In [ ]:
processed_results_rev_holdout = get_top_results_holdout(results_rev_holdout, "accuracy")
display('Amazon Reviews',processed_results_rev_holdout.head())

### Predictions on Test Data

In [ ]:
prediction_results_obesity_holdout, y_pred_obesity_holdout = pred_test_data(obesity_df_test, obesity_best_model_holdout, le_obesity_holdout, "obesity_level_grouped", True)
display('Obesity',prediction_results_obesity_holdout.head())

In [ ]:
prediction_results_depression_holdout, y_pred_depression_holdout = pred_test_data(depression_df_test, depression_best_model_holdout, le_depression_holdout, "depression", True)
display('Depression',prediction_results_depression_holdout.head())

In [41]:
prediction_results_congressional_holdout, y_pred_congressional_holdout = pred_test_data(congressional_df_test, congressional_best_model_holdout, le_congressional_holdout, "class", False)
display('Congressional Voting',prediction_results_congressional_holdout.head())

'Congressional Voting'

,time,parameters
0,0.083271,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."


In [ ]:
prediction_results_rev_holdout, y_pred_rev_holdout = pred_test_data(rev_df_test, rev_best_model_holdout, le_rev_holdout, "class", False)
display('Amazon Reviews',prediction_results_rev_holdout.head())

## Prediction files preparation for Kaggle

In [ ]:
def kaggle_comp_file(pred_df):
  pred_df_final = pred_df[["id", "y_pred"]].rename(columns={"y_pred": "class", "id" : "ID"})
  return pred_df_final

In [42]:
# Congressional Voting
kaggle_submission_congressional_cv = kaggle_comp_file(y_pred_congressional_cv)
kaggle_submission_congressional_holdout = kaggle_comp_file(y_pred_congressional_holdout)

kaggle_submission_congressional_cv.to_csv('kaggle_preds/congressional_random_forest_cv_submission_group39.csv', index=False)
kaggle_submission_congressional_holdout.to_csv('kaggle_preds/congressional_random_forest_holdout_submission_group39.csv', index=False)

In [ ]:
# Amazon Reviews
kaggle_submission_rev_cv = kaggle_comp_file(y_pred_rev_cv)
kaggle_submission_rev_holdout = kaggle_comp_file(y_pred_rev_holdout)

kaggle_submission_rev_cv.to_csv('reviews_random_forest_cv_submission_group39.csv', index=False)
kaggle_submission_rev_holdout.to_csv('reviews_random_forest_holdout_submission_group39.csv', index=False)    

In [ ]:
y_pred_congressional_cv.head()

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-crporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa,y_pred,id
0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,democrat,190
1,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.5,1.0,democrat,285
2,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,republican,251
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,democrat,40
4,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,democrat,91
